In [8]:
import pandas as pd
import socket
import struct
import numpy as np

base_path = r'C:\Users\natna\Downloads\KAIM WEEK-5,6\Improved-detection-of-fraud-cases-for-e-commerce-and-bank-transactions'

# 1. Load Data
fraud_df = pd.read_csv(f'{base_path}\\data\\raw\\Fraud_Data.csv')
ip_df = pd.read_csv(f'{base_path}\\data\\raw\\IpAddress_to_Country.csv')

# 2. Universal IP to Float Conversion
def universal_ip_to_float(val):
    if pd.isna(val):
        return np.nan
    try:
        # Case A: It's already a number (or a string representing a number)
        return float(val)
    except ValueError:
        try:
            # Case B: It's a string IP like '192.168.1.1'
            return float(struct.unpack("!I", socket.inet_aton(str(val).strip()))[0])
        except:
            return np.nan

# Apply conversion
fraud_df['ip_address_int'] = fraud_df['ip_address'].apply(universal_ip_to_float)

# 3. Handle Data Types & Missing Values
# Force numeric type explicitly to avoid the 'O' vs 'float64' MergeError
fraud_df['ip_address_int'] = pd.to_numeric(fraud_df['ip_address_int'], errors='coerce')
ip_df['lower_bound_ip_address'] = pd.to_numeric(ip_df['lower_bound_ip_address'], errors='coerce')

# Drop rows where we have no IP info
fraud_df_clean = fraud_df.dropna(subset=['ip_address_int']).copy()

# Check the ranges again for debugging
print(f"Fraud IP Range: {fraud_df_clean['ip_address_int'].min()} to {fraud_df_clean['ip_address_int'].max()}")
print(f"Country IP Range: {ip_df['lower_bound_ip_address'].min()} to {ip_df['lower_bound_ip_address'].max()}")

# 4. Sort (Crucial for merge_asof)
fraud_df_clean = fraud_df_clean.sort_values('ip_address_int')
ip_df = ip_df.sort_values('lower_bound_ip_address')

# 5. Perform the Merge
merged_df = pd.merge_asof(
    fraud_df_clean, 
    ip_df, 
    left_on='ip_address_int', 
    right_on='lower_bound_ip_address',
    direction='backward'
)

# 6. Final Validation: Match the range and set 'Unknown' for outliers
merged_df['country'] = merged_df.apply(
    lambda x: x['country'] if pd.notnull(x['upper_bound_ip_address']) and x['ip_address_int'] <= x['upper_bound_ip_address'] else 'Unknown', 
    axis=1
)

print(f"\nFinal Merged Row Count: {len(merged_df)}")
print(merged_df[['user_id', 'ip_address', 'country']].head())

Fraud IP Range: 52093.4968949854 to 4294850499.67884
Country IP Range: 16777216.0 to 3758096128.0

Final Merged Row Count: 151112
   user_id     ip_address  country
0    62421   52093.496895  Unknown
1   173212   93447.138961  Unknown
2   242286  105818.501505  Unknown
3   370003  117566.664867  Unknown
4   119824  131423.789042  Unknown


In [9]:
# Convert to datetime
merged_df['signup_time'] = pd.to_datetime(merged_df['signup_time'])
merged_df['purchase_time'] = pd.to_datetime(merged_df['purchase_time'])

# Feature Engineering
merged_df['hour_of_day'] = merged_df['purchase_time'].dt.hour
merged_df['day_of_week'] = merged_df['purchase_time'].dt.dayofweek
merged_df['time_since_signup'] = (merged_df['purchase_time'] - merged_df['signup_time']).dt.total_seconds()

# Save the final cleaned and engineered dataset
output_file = f'{base_path}\\data\\processed\\fraud_data_final.csv'
merged_df.to_csv(output_file, index=False)
print(f"Task 1 Preprocessing Complete! File saved to: {output_file}")

Task 1 Preprocessing Complete! File saved to: C:\Users\natna\Downloads\KAIM WEEK-5,6\Improved-detection-of-fraud-cases-for-e-commerce-and-bank-transactions\data\processed\fraud_data_final.csv


In [10]:
# 1. Convert timestamp columns to datetime objects
merged_df['signup_time'] = pd.to_datetime(merged_df['signup_time'])
merged_df['purchase_time'] = pd.to_datetime(merged_df['purchase_time'])

# 2. Extract Time-Based Features
merged_df['hour_of_day'] = merged_df['purchase_time'].dt.hour
merged_df['day_of_week'] = merged_df['purchase_time'].dt.dayofweek

# 3. Calculate Transaction Velocity (Time since signup)
merged_df['time_since_signup'] = (merged_df['purchase_time'] - merged_df['signup_time']).dt.total_seconds()

# 4. Save the Final Processed Data
output_path = r'C:\Users\natna\Downloads\KAIM WEEK-5,6\Improved-detection-of-fraud-cases-for-e-commerce-and-bank-transactions\data\processed\fraud_data_final.csv'
merged_df.to_csv(output_path, index=False)

print("Feature Engineering Complete!")
print(merged_df[['user_id', 'country', 'hour_of_day', 'time_since_signup']].head())

Feature Engineering Complete!
   user_id  country  hour_of_day  time_since_signup
0    62421  Unknown           10          1763014.0
1   173212  Unknown           17          1084823.0
2   242286  Unknown            8           749320.0
3   370003  Unknown           21          7434634.0
4   119824  Unknown            7          1407619.0
